En esta práctica vamos a trabajar con un conjunto de datos multimodal que contiene señales fisiológicas. Estas señales fueron registradas en 15 estudiantes durante cuatro situaciones distintas.

En el notebook titulado `MLA_1_extraccion_caracteristicas`, realizamos la extracción de características, transformando las señales ECG y EDA en variables que podemos analizar y utilizar junto con modelos de aprendizaje automático.

Este segundo notebook se centra en el desarrollo de un modelo de inteligencia artificial para clasificar la fase en la que se encontraba cada sujeto en función de sus datos biométricos. Disponemos de mediciones correspondientes a las siguientes fases:

* **Fase 0**: no definida; corresponde a una transición entre fases. Estos datos no los utilizaremos en nuestro problema de clasificación y los eliminaremos durante el preprocesamiento.
* **Fase 1**: corresponde a un estado neutral sin emociones.
* **Fase 2**: estado de estrés; los estudiantes se encontraban realizando un examen complejo que no podían resolver.
* **Fase 3**: diversión; los estudiantes estaban visualizando vídeos educativos de humor.
* **Fase 4**: meditación; los estudiantes realizaron un seminario de meditación para registrar sus emociones en un estado de alta relajación.



# 0. Preparación del entorno

Importamos las librerías que vamos a necesitar durante la práctica.

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV, GroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier

Establecemos una semilla de aleatoriedad que se utilizará durante toda la práctica para garantizar la reproducibilidad de los resultados. De este modo, cuando ejecutemos procesos que incluyan componentes aleatorios, se generarán siempre los mismos números aleatorios.

Lo adecuado es almacenar la semilla en una variable y establecerla al inicio del notebook. Además, cada vez que utilicemos una función o método que incorpore aleatoriedad, debemos volver a fijar la semilla o, si es posible, pasarla como argumento de la función (aunque esto no siempre es factible).


In [2]:
SEMILLA_ALEATORIEDAD = 123

In [3]:
np.random.seed(SEMILLA_ALEATORIEDAD)

# 1. Carga de datos

Primero debemos cargar el dataframe generado en el notebook anterior (`MLA_1_extraccion_caracteristicas`). Este archivo se encuentra bajo el nombre `df_caracteristicas_extraidas.csv`.

In [4]:
#from google.colab import drive
#drive.mount('/content/drive')

In [5]:
df = pd.read_csv("./df_caracteristicas_extraidas.csv")

# 2.Análsis exploratorio

El siguiente paso consiste en analizar y entender qué datos tenemos, cómo están representados y qué preprocesamientos son necesarios.


In [6]:
df

,ECG_HRV_MeanNN,ECG_HRV_SDNN,ECG_HRV_SDANN1,ECG_HRV_SDNNI1,ECG_HRV_SDANN2,ECG_HRV_SDNNI2,ECG_HRV_SDANN5,ECG_HRV_SDNNI5,ECG_HRV_RMSSD,ECG_HRV_SDSD,...,ECG_HRV_HFn,ECG_HRV_LnHF,EDA_SCR_Peaks_N,EDA_SCR_Peaks_Amplitude_Mean,EDA_EDA_Tonic_SD,EDA_EDA_Sympathetic,EDA_EDA_SympatheticN,EDA_EDA_Autocorrelation,subject_id,phase_label
0,789.895238,70.512456,NaN,NaN,NaN,NaN,NaN,NaN,42.636858,42.776894,...,0.209272,-4.992914,10.0,0.378945,0.200399,0.012419,0.000567,0.214118,S2,0
1,792.457143,68.756366,NaN,NaN,NaN,NaN,NaN,NaN,41.625807,41.766109,...,0.293790,-4.945348,8.0,0.353897,0.179245,0.008686,0.000343,0.152962,S2,0
2,787.691580,65.694171,NaN,NaN,NaN,NaN,NaN,NaN,44.276343,44.420331,...,0.233815,-4.203081,7.0,0.284399,0.219655,0.056722,0.003884,0.276437,S2,0
3,797.866795,60.212382,NaN,NaN,NaN,NaN,NaN,NaN,45.495254,45.649043,...,0.213022,-4.300312,8.0,0.491586,0.407787,0.037301,0.001715,0.488793,S2,0
4,803.368726,63.630940,NaN,NaN,NaN,NaN,NaN,NaN,50.537579,50.706274,...,0.191188,-4.556492,7.0,0.563820,0.394682,0.027617,0.001055,0.503091,S2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3067,981.711924,128.304832,NaN,NaN,NaN,NaN,NaN,NaN,116.175302,116.661861,...,0.665165,-2.764208,106.0,0.004661,0.009514,0.000525,0.000015,0.354740,S17,0
3068,982.939787,119.986011,NaN,NaN,NaN,NaN,NaN,NaN,107.234009,107.680650,...,0.584359,-3.224464,106.0,0.004483,0.007311,0.000435,0.000012,0.327162,S17,0
3069,953.794286,144.208596,NaN,NaN,NaN,NaN,NaN,NaN,93.342052,93.687411,...,0.684434,-3.615629,34.0,0.007132,0.032350,0.000424,0.000012,0.407286,S17,0
3070,922.292359,155.829681,NaN,NaN,NaN,NaN,NaN,NaN,83.330984,83.650761,...,0.509329,-4.679488,17.0,0.013100,0.049682,0.000477,0.000013,0.667660,S17,0


Nuestro dataset contiene 3072 registros y 43 columnas. Estos registros provienen de 15 estudiantes distintos, cada uno evaluado en cuatro fases diferentes.

De las 43 columnas, 41 corresponden a las señales ECG o EDA registradas durante cada ventana de tiempo. Las columnas que comienzan con `ECG` representan la señal de electrocardiograma, mientras que las que comienzan con `EDA` representan la respuesta electrodérmica.

La columna `subject_id` indica el identificador anonimizado de cada estudiante, y `phase_label` señala la fase en la que se encontraba el estudiante durante esa ventana de tiempo.



Podemos listar el nombre de las columnas mediante el atributo `variable_con_dataframe.columns`

In [7]:
df.columns

Index(['ECG_HRV_MeanNN', 'ECG_HRV_SDNN', 'ECG_HRV_SDANN1', 'ECG_HRV_SDNNI1',
       'ECG_HRV_SDANN2', 'ECG_HRV_SDNNI2', 'ECG_HRV_SDANN5', 'ECG_HRV_SDNNI5',
       'ECG_HRV_RMSSD', 'ECG_HRV_SDSD', 'ECG_HRV_CVNN', 'ECG_HRV_CVSD',
       'ECG_HRV_MedianNN', 'ECG_HRV_MadNN', 'ECG_HRV_MCVNN', 'ECG_HRV_IQRNN',
       'ECG_HRV_SDRMSSD', 'ECG_HRV_Prc20NN', 'ECG_HRV_Prc80NN',
       'ECG_HRV_pNN50', 'ECG_HRV_pNN20', 'ECG_HRV_MinNN', 'ECG_HRV_MaxNN',
       'ECG_HRV_HTI', 'ECG_HRV_TINN', 'ECG_HRV_ULF', 'ECG_HRV_VLF',
       'ECG_HRV_LF', 'ECG_HRV_HF', 'ECG_HRV_VHF', 'ECG_HRV_TP', 'ECG_HRV_LFHF',
       'ECG_HRV_LFn', 'ECG_HRV_HFn', 'ECG_HRV_LnHF', 'EDA_SCR_Peaks_N',
       'EDA_SCR_Peaks_Amplitude_Mean', 'EDA_EDA_Tonic_SD',
       'EDA_EDA_Sympathetic', 'EDA_EDA_SympatheticN',
       'EDA_EDA_Autocorrelation', 'subject_id', 'phase_label'],
      dtype='object')

Podemos obtener estadísticas básicas de todas las variables numéricas utilizando la función `df.describe()`.

In [8]:
df.describe()

,ECG_HRV_MeanNN,ECG_HRV_SDNN,ECG_HRV_SDANN1,ECG_HRV_SDNNI1,ECG_HRV_SDANN2,ECG_HRV_SDNNI2,ECG_HRV_SDANN5,ECG_HRV_SDNNI5,ECG_HRV_RMSSD,ECG_HRV_SDSD,...,ECG_HRV_LFn,ECG_HRV_HFn,ECG_HRV_LnHF,EDA_SCR_Peaks_N,EDA_SCR_Peaks_Amplitude_Mean,EDA_EDA_Tonic_SD,EDA_EDA_Sympathetic,EDA_EDA_SympatheticN,EDA_EDA_Autocorrelation,phase_label
count,3072.000000,3072.000000,0.0,0.0,0.0,0.0,0.0,0.0,3072.000000,3072.000000,...,3072.000000,3072.000000,3072.000000,3072.000000,3072.000000,3072.000000,3072.000000,3072.000000,3072.000000,3072.000000
mean,811.948331,82.624988,NaN,NaN,NaN,NaN,NaN,NaN,58.010723,58.217744,...,0.683570,0.299017,-4.591672,70.100586,0.042571,0.095186,0.003988,0.000113,0.677694,1.245768
std,144.168549,36.873241,NaN,NaN,NaN,NaN,NaN,NaN,41.308385,41.482784,...,0.178260,0.168045,1.109527,49.876748,0.135760,0.202979,0.046260,0.000887,0.202203,1.394440
min,416.162270,16.014875,NaN,NaN,NaN,NaN,NaN,NaN,3.173015,3.177581,...,0.138767,0.016361,-9.424002,1.000000,0.001764,0.000856,0.000002,0.000009,-0.347078,0.000000
25%,722.971861,57.711887,NaN,NaN,NaN,NaN,NaN,NaN,31.430613,31.513102,...,0.556564,0.164662,-5.257582,21.000000,0.003726,0.012777,0.000064,0.000012,0.581712,0.000000
50%,817.334975,74.809749,NaN,NaN,NaN,NaN,NaN,NaN,45.835147,45.991111,...,0.722442,0.264091,-4.544240,62.500000,0.008148,0.028689,0.000213,0.000014,0.748713,1.000000
75%,902.976190,100.273750,NaN,NaN,NaN,NaN,NaN,NaN,73.137595,73.427735,...,0.826701,0.420803,-3.817061,114.000000,0.021041,0.086870,0.000628,0.000026,0.831465,2.000000
max,1220.751105,348.157514,NaN,NaN,NaN,NaN,NaN,NaN,456.277860,458.093679,...,0.982886,0.830038,-2.063290,192.000000,2.165010,3.005413,1.763025,0.032012,0.920337,4.000000


En nuestro caso, queremos predecir la variable `phase_label`. Aunque cada fase está representada con números, estos valores corresponden a categorías. Con la función `df.value_counts()` podemos analizar la distribución de esta variable.


Como la fase es una variable categórica, nos enfrentamos a un problema de clasificación. Durante el preprocesamiento, eliminaremos los datos sin fase, aquellos etiquetados como 0.


In [9]:
df["phase_label"].value_counts()

phase_label
0    1271
1     791
4     414
2     408
3     188
Name: count, dtype: int64

También contamos con la variable `subject_id`, que contiene el identificador de cada estudiante. Esta variable no se utilizará como variable predictora, pero sí es necesaria para realizar evaluaciones justas y realistas. Nos permitirá asegurarnos de que, al dividir nuestro conjunto de datos en entrenamiento y test, y posteriormente al realizar validaciones cruzadas durante la búsqueda de hiperparámetros, no utilicemos datos de un sujeto que el modelo ya haya visto.


In [10]:
df["subject_id"].value_counts()

subject_id
S6     266
S4     235
S3     235
S5     223
S2     220
S17    210
S14    197
S16    196
S13    195
S10    191
S8     189
S15    180
S7     179
S11    179
S9     177
Name: count, dtype: int64

Seguidamente, analizamos la presencia de valores faltantes.

In [11]:
# Calculamos el número de valores faltantes por columna
na_por_columna = df.isna().sum()

# Filtramos las columnas con valores faltantes
columnas_con_na = na_por_columna[na_por_columna > 0]

# Mostramos las columnas con valores faltantes y el número de NA
for columna, na_count in columnas_con_na.items():
    print("Columna: "+str(columna)+ " Número de NA: "+str(na_count))

Columna: ECG_HRV_SDANN1 Número de NA: 3072
Columna: ECG_HRV_SDNNI1 Número de NA: 3072
Columna: ECG_HRV_SDANN2 Número de NA: 3072
Columna: ECG_HRV_SDNNI2 Número de NA: 3072
Columna: ECG_HRV_SDANN5 Número de NA: 3072
Columna: ECG_HRV_SDNNI5 Número de NA: 3072
Columna: ECG_HRV_ULF Número de NA: 3072
Columna: ECG_HRV_VLF Número de NA: 3072


El código anterior muestra que, debido al tamaño de ventana seleccionado, la librería NeuroKit no ha podido extraer algunas características en determinadas ventanas. Como resultado, se han generado columnas vacías que debemos eliminar durante el preprocesamiento.


# 3.Preprocesamiento

La fase de preprocesamiento incluye todos los pasos necesarios para preparar nuestros datos, de manera que puedan ser utilizados por los modelos de machine learning.

## 3.1.Limpieza de datos

El primer paso consiste en eliminar los datos que no vamos a utilizar. En nuestro caso, es necesario eliminar las columnas que quedaron vacías debido a problemas durante la extracción de características.

In [12]:
print(df.shape)  # antes
df = df.dropna(axis=1, how='all')
print(df.shape)  # después


(3072, 43)
(3072, 35)


Seleccionamos sólo los datos recogidos en las fases 1, 2, 3 y 4. Por lo tanto, no utilizaremos los datos etiquetados como 0.

In [13]:
df = df[df["phase_label"].isin([1, 2, 3, 4])]

## 3.2.División del conjunto de datos en entrenamiento y test

En este paso, debemos dividir nuestro conjunto de datos de forma aleatoria en entrenamiento y test. Para ello, utilizaremos la función `train_test_split` de sklearn. Esta función recibe como primer argumento los datos a dividir, en `test_size` la proporción de datos que queremos dejar en el conjunto de test, y en `random_state` la semilla de aleatoriedad para garantizar que la división sea reproducible. La función devuelve primero los datos de entrenamiento y luego los del conjunto de test.

Para esta práctica, es necesario realizar una división por sujetos, de modo que el 70% de los sujetos se asignen al conjunto de entrenamiento y el 30% restante al conjunto de test.



In [14]:
subjects = df["subject_id"].unique()

subjects_train, subjects_test = train_test_split(
    subjects, test_size=0.3, random_state=SEMILLA_ALEATORIEDAD
)

dataset_train = df[df["subject_id"].isin(subjects_train)]
dataset_test = df[df["subject_id"].isin(subjects_test)]


## 3.4 División entre variables predictoras y variable a predicer

El siguiente paso consiste en separar las variables predictoras (las características extraídas con NeuroKit) de la variable a predecir (la fase en la que se recogió cada señal). Dado que estamos combinando las características extraídas de las señales ECG y EDA, estamos trabajando con datos multimodales. Para ello, hemos concatenado las características de ambas señales, aplicando una fusión temprana o a nivel de característica.


En una variable adicional, independiente del conjunto de variables predictoras, conservamos los identificadores de los sujetos. Necesitaremos estos identificadores para realizar validaciones cruzadas durante la búsqueda de hiperparámetros, asegurándonos de que los datos de un mismo sujeto no se separen en diferentes pliegues.


In [15]:
train_x = dataset_train.loc[:, ~dataset_train.columns.isin(["phase_label", "subject_id"])]
train_y = dataset_train.loc[:, "phase_label"]
train_subject = dataset_train.loc[:, "subject_id"]


test_x = dataset_test.loc[:, ~dataset_test.columns.isin(["phase_label", "subject_id"])]
test_y = dataset_test.loc[:, "phase_label"]
test_subject = dataset_test.loc[:, "subject_id"]

# 4.Configuración y selección de modelos

Este paso se centra en probar diferentes modelos y configuraciones (hiperparámetros) de cada modelo. Para simplificar esta práctica sólo haremos una búsqueda de hiperámetros del modelo Random Forest.

Para las validaciones cruzadas, estableceremos los grupos según los sujetos, en nuestro caso, los estudiantes. Para ello, utilizaremos la función `GroupKFold` de `sklearn`, que integraremos en nuestra búsqueda de hiperparámetros con Grid Search.

De esta manera, nos aseguramos de que todas las muestras de un mismo estudiante permanezcan juntas, ya sea en el conjunto de entrenamiento o en el de validación, durante cada pliegue. Esto evita que el modelo vea datos de un sujeto durante el entrenamiento y luego los utilice para validar, garantizando evaluaciones más realistas y justas.

Aquí tienes un ejemplo de la función `GroupKFold` de `sklearn`. En él se puede observar cómo los datos de un mismo grupo no se mezclan entre diferentes pliegues.

In [16]:
# Datos de ejemplo: 6 muestras con 1 característica
#     0     1    2    3    4    5   índice
X = [[1], [2], [3], [4], [5], [6]]
y = [0,    0,   1,   1,   0,   1]

# Grupos: cada sujeto tiene 2 muestras
# Índice:  0  1  2  3  4  5
groups = [ 1, 1, 2, 2, 3, 3]

# Creamos 2 pliegues
gkf = GroupKFold(n_splits=2)

for train_idx, val_idx in gkf.split(X, y, groups):
    print("Entrenamiento:", train_idx, "Validación:", val_idx)

Entrenamiento: [2 3] Validación: [0 1 4 5]
Entrenamiento: [0 1 4 5] Validación: [2 3]


In [17]:
# Validación cruzada sin mezclar sujetos (estudiantes)
cv = GroupKFold(n_splits=5)

In [18]:
# Definimos la métrica para optimizar
OPT = "F1"

# Modelo
model = RandomForestClassifier(random_state=SEMILLA_ALEATORIEDAD)

# Grid de hiperparámetros
param_grid = {
    'n_estimators': [100, 200, 400, 600],  
    'max_depth': [None, 10, 20],             
    'min_samples_split': [2, 5, 10],                                 
    'max_features': ['sqrt', 'log2', None],    
    'bootstrap': [True, False]
}


# Definir GridSearchCV
grid_search = GridSearchCV(
    model,
    param_grid,
    cv=cv,
    scoring={"F1": "f1_macro", "Accuracy": "accuracy"},
    refit=OPT,
    n_jobs=-1
)

# Realizamos la búsqueda de hiperparámetros
np.random.seed(SEMILLA_ALEATORIEDAD)
grid_search.fit(train_x, train_y, groups=train_subject)   # ← Aquí los subject_id

# Resultados
print("Mejores hiperparámetros:", grid_search.best_params_)
print("Mejor puntuación (" + OPT + "):", grid_search.best_score_)



Mejores hiperparámetros: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 600}
Mejor puntuación (F1): 0.46092827137188747


También podemos mostrar estos rendimientos en un dataframe:

In [19]:
grid_search.best_params_

{'bootstrap': True,
 'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_split': 5,
 'n_estimators': 600}

# 5.Entrenamiento del mejor modelo

A continuación, entrenamos la configuración y el modelo que obtuvo el mejor rendimiento en la búsqueda de hiperparámetros, utilizando todo el conjunto de entrenamiento y sin aplicar validaciones cruzadas. Para ello, es necesario obtener los hiperparámetros del mejor modelo de la búsqueda mediante `grid_search.best_params_`.


In [20]:
# Creamos un modelo con los hiperparámetros de la mejor configuración de la búsqueda de hiperparámetros
best_model = RandomForestClassifier(**grid_search.best_params_)

# Entrenamos el modelo con todo el conjunto de datos
np.random.seed(SEMILLA_ALEATORIEDAD)
best_model.fit(train_x, train_y)



RandomForestClassifier(max_depth=10, min_samples_split=5, n_estimators=600)

# 6.Evaluación del mejor modelo en test

Finalmente, calculamos el rendimiento del modelo con los sujetos reservados para test. 

In [21]:
# Obtenemos las predicciones del modelo en test
predictions = best_model.predict(test_x)

# Cálculo de métricas
f1 = f1_score(test_y, predictions, average='macro')
acc = accuracy_score(test_y, predictions)

print("F1 (macro):", f1)
print("Accuracy:", acc)


F1 (macro): 0.5341241750573522
Accuracy: 0.5932773109243697


Podemos observar que el rendimiento es bajo. Sin embargo, cabe destacar las complejidades técnicas del problema. Estamos utilizando señales fisiológicas, las cuales no se comportan de forma homogénea entre todas las personas. Además, los datos fueron recogidos mediante un caso de estudio, por lo que existe cierto ruido en los sensores, lo que dificulta la generalización del modelo.

A pesar de estas limitaciones, pueden alcanzarse exactitudes (accuracy) en torno al 90% utilizando procesamientos más complejos y completos. Por ejemplo, procesando todas las señales, extrayendo un mayor número de características y aplicando otros modelos y técnicas de preprocesamiento.




# Ejercicios entregables

Utilizando los dos notebooks de la práctica 4 sobre analíticas de aprendizaje multimodales, debes realizar las siguientes tareas:

1. Extiende el análisis a otras señales disponibles en nuestro conjunto de datos o incrementa el número de características extraídas de las señales ECG y EDA. Para procesar nuevas señales puedes usar funciones disponibles en la documentación de la librería [NeuroKit2](https://neuropsychology.github.io/NeuroKit/functions/index.html), buscar librerías externas o realizar tu propia implementación.

2. Realiza un escalado de las variables predictoras.

3. Amplía los hiperparámetros analizados en la búsqueda de hiperparámetros del modelo Random Forest y evalúa al menos otros tres modelos diferentes.
